# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata

print(f"Dataset name: {meta.name}\n")
print(f"Description: {meta.description}\n")
print(f"Published: {meta.datePublished}")


## 2. Data Overview
Review available record sets, fields, and their IDs using the Croissant metadata.

In [ ]:
# List all record sets and their fields with @id references
record_sets = meta.recordSets
if not record_sets:
    # Try extracting via dataset interface if metadata property is missing
    from mlcroissant.converters.croissant_to_metadata import CroissantToMetadata
    croissant_json = dataset._croissant_json
    croissant_meta = CroissantToMetadata(croissant_json).metadata
    record_sets = croissant_meta.recordSets

if not record_sets:
    raise RuntimeError('No record sets found in metadata!')

for rs in record_sets:
    print(f"RecordSet name: {rs.name} (@id={rs.id})")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id={field.id}) type={field.dataType}")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.
The code below loads all record sets and stores each as a pandas DataFrame, indexed by the record set `@id`.

In [ ]:
# Gather all record set IDs
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    try:
        # Each record is a dictionary with '@id' keys as columns
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for {record_set_id}")
    except Exception as e:
        print(f"Failed to load record set {record_set_id}: {e}")

# For demonstration, pick the first record set
if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print(f"\nExample RecordSet columns (@id={example_record_set_id}):")
    print(dataframes[example_record_set_id].columns.tolist())
    print("\nPreview:")
    display(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll:
- Select a numeric field from the chosen record set (referenced by `@id`).
- Filter records, normalize values, and group by a categorical field.

In [ ]:
import numpy as np
# Identify a numeric field to analyze (by @id) and a group field (by @id)
# You should check the output above for field @ids and types; here we'll assume commonly-used ones used in proteomics
rs = next((rs for rs in record_sets if rs.id == example_record_set_id), None)

# Suggest likely numeric fields: search those with type 'Float' or 'Integer'
numeric_fields = [f.id for f in rs.fields if f.dataType.lower() in ['float', 'integer', 'number']]
group_fields = [f.id for f in rs.fields if f.dataType.lower() not in ['float', 'integer', 'number']]

if not numeric_fields:
    raise RuntimeError('No numeric fields found to analyze')

numeric_field_id = numeric_fields[0]  # e.g., '@id': 'abundance', or similar
print(f'Selected numeric field (@id): {numeric_field_id}')

df = dataframes[example_record_set_id]
# Convert to numeric if necessary
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# For demo, set threshold as median
threshold = df[numeric_field_id].median()

filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold} (n={len(filtered_df)}):")
display(filtered_df.head())

# Normalize the selected numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Optionally group by the first non-numeric field (e.g., accession or category)
grouped_field_id = group_fields[0] if group_fields else None
if grouped_field_id and grouped_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(grouped_field_id)[numeric_field_id].mean()
    print(f"Grouped mean {numeric_field_id} by {grouped_field_id}:")
    display(grouped.head())

## 5. Visualization
Visualize the distribution of a numeric field, and the grouping if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If grouped, show barplot for top 10 group means
if grouped_field_id and grouped_field_id in df.columns:
    top_groups = df.groupby(grouped_field_id)[numeric_field_id].mean().sort_values(ascending=False).head(10)
    plt.figure(figsize=(10,5))
    sns.barplot(y=top_groups.index, x=top_groups.values, orient='h')
    plt.xlabel(f'Mean {numeric_field_id}')
    plt.title(f'Top 10 {grouped_field_id} by mean {numeric_field_id}')
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load a rich mass spectrometry dataset using the Croissant specification and the `mlcroissant` library. We programmatically explored available record sets and fields by `@id`, loaded data into DataFrames, performed filtering and normalization on numeric features, and visualized distributions and group statistics. This workflow can be adapted for deeper proteomics analysis, biomarker studies, or automated FAIR^2 dataset exploration with transparent provenance using the Croissant ecosystem.